# Goal statement:
Does Class II/III latency show the same sharp 2013→2019 decline-then-plateau shape, or is the plateau unique to Class I

## Subgoal:
Did FDA optimize the Class I pipeline specifically, leaving lower-severity recalls untouched

# Environment Setup & Storage Mount

In [32]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Libraries Installation & Imports

In [33]:
!pip install pandera -q

import glob # used to merge multiple CSV files from one folder in Google Drive
import os
import numpy as np
import pandas as pd
import pandera.pandas as pa
from pandera.pandas import Column, Check

# Data Processing

## Lean Six Sigma Palette Configuration

Every chart below draws from one shared palette so the same color always carries the same meaning:


*   Blue — neutral process data / central tendency (no judgment implied)
*   Grey — contextual / denominator data (e.g., raw volume) — background information, not a signal
* Amber — warning zone / approaching an out-of-spec condition
* Red — out-of-spec / critical — reserved strictly for confirmed threshold breaches
* Dark neutral — statistical overlays (trend/fit lines). A model artifact, not a verdict

In [34]:
# Shared Lean Six Sigma color palette
SIXSIGMA_PALETTE = {
    'blue':   '#1F6FB2',  # neutral process data / central tendency
    'grey':   '#8C8C8C',  # contextual / denominator data (no signal)
    'amber':  '#FFC000',  # warning zone
    'red':    '#C00000',  # out-of-spec / critical
    'dark':   '#404040',  # statistical overlay (trend/fit lines)
    'green':  '#548235',  # in-control / meets target (reserved, unused for now)
}

## File Path Matching & CSV Ingestion

In [35]:
# Define folder path
folder_path = (
    "/content/drive/MyDrive/FDA recall data analysis/Class 2 3 Latency/Raw data"
)
# Get the CSV files in the folder
all_files = glob.glob(f"{folder_path}/*.csv")
print(f"Loaded {len(all_files)} files.")

# Read CSV files and assign source file name as the last column (used for tracability)
df_list = [
    pd.read_csv(file).assign(source_file=os.path.basename(file))
    for file in all_files
]

# combined_df = pd.concat(df_list, ignore_index=True)

Loaded 22 files.


## Pre-Merge Schema Validation Audit

In [36]:
# Standardize column names across all DataFrames inline
for df in df_list:
    df.columns = df.columns.str.strip().str.lower()

# Extract schema mapping and unique column sets
schema_map = {
    os.path.basename(f): set(df.columns) for f, df in zip(all_files, df_list)
}
all_columns = set.union(*schema_map.values())
common_columns = set.intersection(*schema_map.values())
missing_anywhere = all_columns - common_columns

print(f"Total Unique Columns Across Files: {len(all_columns)}")
print(f"Columns Common to ALL Files: {len(common_columns)}")
print(f"Columns Missing in at Least One File: {len(missing_anywhere)}\n")

# Print Schema Divergence by File
print ("-" * 60)
print("SCHEMA DIVERGENCE BY FILE")
print ("-" * 60)

for file_name, cols in schema_map.items():
    missing = all_columns - cols
    extra = cols - common_columns
    if missing or extra:
        print(f"\n--- File: {file_name} ---")
        if missing:
            print(f"  Missing columns ({len(missing)}): {sorted(list(missing))}")
        if extra:
            print(f"  Extra columns ({len(extra)}): {sorted(list(extra))}")

Total Unique Columns Across Files: 29
Columns Common to ALL Files: 26
Columns Missing in at Least One File: 3

------------------------------------------------------------
SCHEMA DIVERGENCE BY FILE
------------------------------------------------------------

--- File: Class 1 Drug recalls Jan 2024 to 12 July 2026.csv ---
  Missing columns (3): ['more code info', 'more code info.1', 'more code info.2']

--- File: Class 1 Drug recalls Jan 2013 to Dec 2018.csv ---
  Missing columns (3): ['more code info', 'more code info.1', 'more code info.2']

--- File: Class 1 Drug recalls Jan 2019 to Dec 2023.csv ---
  Missing columns (3): ['more code info', 'more code info.1', 'more code info.2']

--- File: 08 Jun 2012 to Dec 2012.csv ---
  Missing columns (3): ['more code info', 'more code info.1', 'more code info.2']

--- File: Jan 2013 to Jun 2013.csv ---
  Missing columns (3): ['more code info', 'more code info.1', 'more code info.2']

--- File: Jul 2013 to Dec 2013.csv ---
  Missing columns (3)

## Concatenation & Null Standardization Audit

In [37]:
# Combine all DataFrames
combined_df = pd.concat(df_list, ignore_index=True)

# Convert empty strings, spaces, and common string flags to true NaNs
df_clean = combined_df.replace(r"^\s*$", np.nan, regex=True).replace(
    ["N/A", "n/a", "null", "None", "NONE", "UNKNOWN", "unknown"], np.nan
)

# Audit missing data before imputation
missing_summary = pd.DataFrame(
    {
        "Data Type": df_clean.dtypes,
        "Total Rows": len(df_clean),
        "Missing Count": df_clean.isna().sum(),
        "Missing (%)": (df_clean.isna().mean() * 100).round(2),
    }
)

print("=== MISSING DATA AUDIT ===")
print(
    missing_summary[missing_summary["Missing Count"] > 0].sort_values(
        by="Missing Count", ascending=False
    )
)

=== MISSING DATA AUDIT ===
                                                 Data Type  Total Rows  \
more code info.2                                    object       17726   
more code info.1                                    object       17726   
more code info                                      object       17726   
last modified date                                  object       17726   
address2                                            object       17726   
termination date                                    object       17726   
product quantity                                    object       17726   
state/province                                      object       17726   
initial firm notification of consignee or public    object       17726   
voluntary/mandated                                  object       17726   
code info                                           object       17726   
recall number                                       object       17726   
country    

## Multi-Type Data Imputation & Cleanup

In [38]:
print ("-" * 60)
# 1. STANDARDIZE MISSING PLACEHOLDERS
print ("-" * 60)
# Convert empty strings, spaces, and common string flags to true NaNs
df_clean = combined_df.replace(r"^\s*$", np.nan, regex=True).replace(
    ["N/A", "n/a", "null", "None", "NONE", "UNKNOWN", "unknown"], np.nan
)

print ("-" * 60)
# 2. AUDIT MISSING DATA BEFORE IMPUTATION
print ("-" * 60)
missing_summary = pd.DataFrame(
    {
        "Data Type": df_clean.dtypes,
        "Total Rows": len(df_clean),
        "Missing Count": df_clean.isna().sum(),
        "Missing (%)": (df_clean.isna().mean() * 100).round(2),
    }
)

print("=== MISSING DATA AUDIT ===")
print(
    missing_summary[missing_summary["Missing Count"] > 0].sort_values(
        by="Missing Count", ascending=False
    )
)
print("\n" + "=" * 50 + "\n")

print ("-" * 60)
# 3. APPLY CATEGORICAL / TEXT IMPUTATIONS
print ("-" * 60)
# For text/metadata columns (e.g., lot numbers, firm names, reason for recall),
# fill missing values with explicit labels rather than dropping rows.
text_cols = df_clean.select_dtypes(include=["object"]).columns.tolist()

for col in text_cols:
    if "lot" in col.lower():
        # Specific flag for lot number columns missing in newer files
        df_clean[col] = df_clean[col].fillna("Not Recorded")
    else:
        # General placeholder for other missing metadata text
        df_clean[col] = df_clean[col].fillna("Unknown")

print ("-" * 60)
# 4. APPLY DATE / TEMPORAL IMPUTATIONS
print ("-" * 60)
# For missing date fields, forward-fill or back-fill within sorted groups (or dataset-wide)
date_cols = [
    c for c in df_clean.columns if "date" in c.lower() or "time" in c.lower()
]

for col in date_cols:
    if col in df_clean.columns:
        # Ensure column is datetime format before filling
        df_clean[col] = pd.to_datetime(df_clean[col], errors="coerce")
        # Example forward-fill along temporal sequence (optional)
        # df_clean[col] = df_clean[col].ffill()

print ("-" * 60)
# 5. APPLY NUMERIC IMPUTATIONS
print ("-" * 60)
# For numeric metrics (e.g., quantities, counts), fill with median or zero
numeric_cols = df_clean.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

for col in numeric_cols:
    if df_clean[col].isna().sum() > 0:
        # Use median to avoid skew from outliers
        median_val = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_val)

print ("-" * 60)
# 6. VERIFY CLEANUP RESULTS
print ("-" * 60)
remaining_nulls = df_clean.isna().sum().sum()
print(
    f"Imputation Complete. Total Remaining Nulls in Entire Dataset: {remaining_nulls}"
)

------------------------------------------------------------
------------------------------------------------------------
------------------------------------------------------------
------------------------------------------------------------
=== MISSING DATA AUDIT ===
                                                 Data Type  Total Rows  \
more code info.2                                    object       17726   
more code info.1                                    object       17726   
more code info                                      object       17726   
last modified date                                  object       17726   
address2                                            object       17726   
termination date                                    object       17726   
product quantity                                    object       17726   
state/province                                      object       17726   
initial firm notification of consignee or public    object     

/tmp/ipykernel_1533/394681647.py:55: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_clean[col] = pd.to_datetime(df_clean[col], errors="coerce")
/tmp/ipykernel_1533/394681647.py:55: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_clean[col] = pd.to_datetime(df_clean[col], errors="coerce")
/tmp/ipykernel_1533/394681647.py:55: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_clean[col] = pd.to_datetime(df_clean[col], errors="coerce")
